# GlueX analysis tutorial: $\gamma p \to \eta\pi^+ n$ in Python

The Python half of the tutorial in `README.md`, built on `flat_pipneta__B4_v6.root` (one flat TTree, `pipneta__B4`). Each section matches one exercise; the same code is in `python/exN_*.py`.

Run the cells in order with **Shift+Enter**. If a cell fails, read the last line of the error first.

**Setup.** You need `uproot`, `numpy`, `matplotlib` and `scipy` (`pip install uproot matplotlib scipy`). Change `filename` below if your copy of the ROOT file is somewhere else. Each exercise reads the columns it needs from the 810 MB file, so a cell can take 10 to 30 seconds.

In [ ]:
import os
import numpy as np
import uproot
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2 as chi2_dist
%matplotlib inline

filename = "../flat_pipneta__B4_v6.root"
os.makedirs("plots", exist_ok=True)

## Exercise 1: What is inside the file?

Lists the tree, its 23 branches and their types, shows the first 5 entries, and counts entries passing simple conditions. Expect 6,091,343 entries.

In [ ]:
# Use the path given on the command line, or the default one.

f = uproot.open(filename)

print("=== Contents of the file ===")
print(f.classnames())           # {name: type} for every object in the file

tree = f["pipneta__B4"]
print("\n=== The tree ===")
print("Tree name:         ", tree.name)
print("Number of entries: ", tree.num_entries)
print("Number of branches:", len(tree.keys()), "\n")

# Each branch with its type: double = decimal number, int32_t = whole number
for name, branch in tree.items():
    print(f"  {name:20s} {branch.typename}")

print("\n=== First 5 entries ===")
cols = ["Beam_E", "Chi2_NDF", "Mandelstam_t", "MissingMassSquared", "accidweight", "pol"]
first = tree.arrays(cols, entry_stop=5, library="np")   # dict of numpy arrays
print("Row " + "".join(f"{c[:12]:>14s}" for c in cols))
for i in range(5):
    print(f"{i:3d} " + "".join(f"{first[c][i]:14.6g}" for c in cols))

print("\n=== Counting with conditions ===")
# Read only the branches we need (reading all 23 would use a lot of memory).
d = tree.arrays(["accidweight", "Chi2_NDF", "pol"], library="np")
# (array == 1) is an array of True/False; summing it counts the Trues.
print("accidweight == 1 (in-time beam photon):", np.sum(d["accidweight"] == 1))
print("accidweight <  0 (out-of-time photon): ", np.sum(d["accidweight"] < 0))
print("Chi2_NDF < 10:                         ", np.sum(d["Chi2_NDF"] < 10))

print("\nEntries for each value of pol:")
values, counts = np.unique(d["pol"], return_counts=True)
for v, n in zip(values, counts):
    print(f"  pol = {v}: {n}")
plt.show()

## Exercise 2: Draw labelled histograms

Beam energy, and the kinematic-fit $\chi^2$/ndf with and without a cut. Every plot needs a title and axis labels with units.

In [ ]:
tree = uproot.open(filename)["pipneta__B4"]
d = tree.arrays(["Beam_E", "Chi2_NDF"], library="np")

# np.histogram counts entries in each bin: same bins as the C++ version.
beam_counts, beam_edges = np.histogram(d["Beam_E"], bins=200, range=(8.0, 12.0))
chi_all, chi_edges = np.histogram(d["Chi2_NDF"], bins=100, range=(0.0, 50.0))
cut = d["Chi2_NDF"] < 10                      # True/False for every entry
chi_cut, _ = np.histogram(d["Chi2_NDF"][cut], bins=100, range=(0.0, 50.0))

print("Entries in hBeam:   ", len(d["Beam_E"]))
print("Entries in hChi2All:", len(d["Chi2_NDF"]), f" (inside 0-50: {chi_all.sum()})")
print("Entries in hChi2Cut:", cut.sum())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.stairs(beam_counts, beam_edges, color="tab:blue", linewidth=1.5)
ax1.set_title("Photon beam energy")
ax1.set_xlabel(r"$E_\gamma$ (GeV)")
ax1.set_ylabel("Entries / 20 MeV")

ax2.stairs(chi_all, chi_edges, color="black", linewidth=1.5, label="All entries")
ax2.stairs(chi_cut, chi_edges, color="tab:red", fill=True, alpha=0.4,
           label=r"$\chi^2$/ndf < 10")
ax2.set_yscale("log")
ax2.set_title(r"Kinematic-fit $\chi^2$/ndf")
ax2.set_xlabel(r"$\chi^2$/ndf")
ax2.set_ylabel("Entries / 0.5")
ax2.legend()

fig.tight_layout()
fig.savefig("plots/py_ex2_draw.png", dpi=100)
print("Saved plots/py_ex2_draw.png")
plt.show()

## Exercise 3: Your own cuts and a cut flow

Compute the $\eta\pi^+$ mass from the four-momenta, apply three cuts in sequence, count what survives (raw and accidental-weighted). Change the numbers under YOUR CUTS and rerun.

In [ ]:
# ---- YOUR CUTS ------------------------------------------------------
max_chi2ndf = 10.0    # keep Chi2_NDF below this
max_minus_t = 1.0     # keep -t below this (GeV^2)
max_abs_mm2 = 0.05    # keep |MissingMassSquared| below this (GeV^2)

tree = uproot.open(filename)["pipneta__B4"]
branches = ["PiPlus_Px", "PiPlus_Py", "PiPlus_Pz", "PiPlus_E",
            "Eta_Px", "Eta_Py", "Eta_Pz", "Eta_E",
            "Chi2_NDF", "Mandelstam_t", "MissingMassSquared", "accidweight"]
print(f"Reading {tree.num_entries} entries...")
d = tree.arrays(branches, library="np")

# Invariant mass of the eta pi+ pair: M^2 = (E1+E2)^2 - |p1+p2|^2
E  = d["PiPlus_E"]  + d["Eta_E"]
px = d["PiPlus_Px"] + d["Eta_Px"]
py = d["PiPlus_Py"] + d["Eta_Py"]
pz = d["PiPlus_Pz"] + d["Eta_Pz"]
mass = np.sqrt(E**2 - px**2 - py**2 - pz**2)
w = d["accidweight"]

# Each step keeps what the previous step kept AND passes one more cut.
step0 = np.ones(len(w), dtype=bool)                        # everything
step1 = step0 & (d["Chi2_NDF"] < max_chi2ndf)
step2 = step1 & (-d["Mandelstam_t"] < max_minus_t)
step3 = step2 & (np.abs(d["MissingMassSquared"]) < max_abs_mm2)
steps = [step0, step1, step2, step3]
names = ["All entries", "Chi2_NDF cut", "-t cut", "|MM2| cut"]
labels = ["All entries", rf"$\chi^2$/ndf < {max_chi2ndf:g}",
          rf"$-t$ < {max_minus_t:g} GeV$^2$", rf"|MM$^2$| < {max_abs_mm2:g} GeV$^2$"]

print(f"\n{'Step':26s}{'Entries':>12s}{'% kept':>10s}{'Weighted':>14s}")
weighted = []
for name, keep in zip(names, steps):
    n = keep.sum()
    sw = w[keep].sum()                     # sum of weights = accidental-subtracted count
    weighted.append(sw)
    print(f"{name:26s}{n:12d}{100 * n / len(w):9.1f}%{sw:14.0f}")

# Mass histogram after all cuts, weighted. The error on a weighted bin is
# sqrt(sum of weights squared) - the same thing ROOT's Sumw2() does.
edges = np.linspace(0.7, 2.5, 91)                     # 90 bins of 20 MeV
counts, _ = np.histogram(mass[step3], bins=edges, weights=w[step3])
sumw2, _ = np.histogram(mass[step3], bins=edges, weights=w[step3] ** 2)
errors = np.sqrt(sumw2)
centres = 0.5 * (edges[1:] + edges[:-1])
print(f"\nWeighted entries in the mass plot: {counts.sum():.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar(range(4), weighted, width=1.0, color="lightsteelblue", edgecolor="navy")
ax1.set_xticks(range(4), labels, rotation=20)
ax1.set_yscale("log")
ax1.set_ylim(1e4, None)
ax1.set_ylabel("Weighted entries")
ax1.set_title("Cut flow (accidental-subtracted)")

ax2.errorbar(centres, counts, yerr=errors, fmt="o", ms=3, color="black")
ax2.set_title(r"$\eta\pi^+$ mass after all cuts")
ax2.set_xlabel(r"M($\eta\pi^+$) (GeV/$c^2$)")
ax2.set_ylabel("Weighted entries / 20 MeV")
ax2.set_ylim(0, None)

fig.tight_layout()
fig.savefig("plots/py_ex3_cutflow.png", dpi=100)
print("Saved plots/py_ex3_cutflow.png")
plt.show()

## Exercise 4: What does a cut do? And what are accidentals?

Compare shapes before/after the $\chi^2$ cut (normalized), and see prompt vs accidental vs subtracted.

In [ ]:
tree = uproot.open(filename)["pipneta__B4"]
d = tree.arrays(["PiPlus_Px", "PiPlus_Py", "PiPlus_Pz", "PiPlus_E",
                 "Eta_Px", "Eta_Py", "Eta_Pz", "Eta_E",
                 "Chi2_NDF", "accidweight"], library="np")

E  = d["PiPlus_E"]  + d["Eta_E"]
px = d["PiPlus_Px"] + d["Eta_Px"]
py = d["PiPlus_Py"] + d["Eta_Py"]
pz = d["PiPlus_Pz"] + d["Eta_Pz"]
mass = np.sqrt(E**2 - px**2 - py**2 - pz**2)
w = d["accidweight"]
chi2cut = d["Chi2_NDF"] < 10
prompt = w == 1
accid = w < 0

edges = np.linspace(0.7, 2.5, 91)

def hist(mask, weights):
    """Weighted histogram of the mass for the entries where mask is True."""
    counts, _ = np.histogram(mass[mask], bins=edges, weights=weights[mask])
    return counts

everything = np.ones(len(w), dtype=bool)
h_nocut  = hist(everything, w)
h_cut    = hist(chi2cut, w)
h_prompt = hist(chi2cut & prompt, np.ones(len(w)))     # each prompt entry counts 1
h_accid  = hist(chi2cut & accid, -w)                   # -weight is positive
h_sub    = hist(chi2cut, w)                            # prompt minus accidental

print(f"Weighted sum, no cut:              {h_nocut.sum():.0f}")
print(f"Weighted sum, chi2/ndf < 10:       {h_cut.sum():.0f}")
print(f"Fraction kept by the cut:          {h_cut.sum() / h_nocut.sum():.4f}")
print(f"Prompt entries (chi2/ndf < 10):    {h_prompt.sum():.0f}")
print(f"Accidental estimate:               {h_accid.sum():.1f}")
print(f"Prompt - accidental:               {h_sub.sum():.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
xlabel = r"M($\eta\pi^+$) (GeV/$c^2$)"

ax1.stairs(h_nocut / h_nocut.sum(), edges, color="black", lw=1.5, label="No cut")
ax1.stairs(h_cut / h_cut.sum(), edges, color="tab:red", lw=1.5, label=r"$\chi^2$/ndf < 10")
ax1.set_title(r"Effect of the $\chi^2$/ndf cut")
ax1.set_xlabel(xlabel)
ax1.set_ylabel("Fraction / 20 MeV")
ax1.set_ylim(0, None)
ax1.legend(loc="lower right")

ax2.stairs(h_prompt, edges, color="tab:blue", lw=1.5, label="In-time (prompt)")
ax2.stairs(h_accid, edges, color="gray", fill=True, alpha=0.5, label="Accidental estimate")
ax2.stairs(h_sub, edges, color="tab:red", lw=1.5, label="Prompt − accidental")
ax2.set_title(r"Accidental subtraction ($\chi^2$/ndf < 10)")
ax2.set_xlabel(xlabel)
ax2.set_ylabel("Entries / 20 MeV")
ax2.set_ylim(0, 1.4 * h_prompt.max())
ax2.legend(loc="upper right")

fig.tight_layout()
fig.savefig("plots/py_ex4_compare.png", dpi=100)
print("Saved plots/py_ex4_compare.png")
plt.show()

## Exercise 5: A 2D plot and its projections

$M(\eta\pi^+)$ against $-t$, and projections for two $-t$ slices.

In [ ]:
tree = uproot.open(filename)["pipneta__B4"]
d = tree.arrays(["PiPlus_Px", "PiPlus_Py", "PiPlus_Pz", "PiPlus_E",
                 "Eta_Px", "Eta_Py", "Eta_Pz", "Eta_E",
                 "Chi2_NDF", "MissingMassSquared", "Mandelstam_t", "accidweight"],
                library="np")

E  = d["PiPlus_E"]  + d["Eta_E"]
px = d["PiPlus_Px"] + d["Eta_Px"]
py = d["PiPlus_Py"] + d["Eta_Py"]
pz = d["PiPlus_Pz"] + d["Eta_Pz"]
mass = np.sqrt(E**2 - px**2 - py**2 - pz**2)
minus_t = -d["Mandelstam_t"]
keep = (d["Chi2_NDF"] < 10) & (np.abs(d["MissingMassSquared"]) < 0.05)

x_edges = np.linspace(0.7, 2.5, 91)    # 90 mass bins
y_edges = np.linspace(0.0, 2.0, 41)    # 40 -t bins of 0.05 GeV^2
h2, _, _ = np.histogram2d(mass[keep], minus_t[keep], bins=[x_edges, y_edges],
                          weights=d["accidweight"][keep])
# h2[i, j] = content of mass bin i, -t bin j

# Projection = add up the -t bins inside a range (axis=1 is the -t direction)
y_centres = 0.5 * (y_edges[1:] + y_edges[:-1])
p_low  = h2[:, (y_centres > 0.0) & (y_centres < 0.5)].sum(axis=1)
p_high = h2[:, (y_centres > 0.5) & (y_centres < 1.0)].sum(axis=1)

print(f"Weighted entries in the 2D plot:     {h2.sum():.0f}")
print(f"Weighted entries with 0 < -t < 0.5:   {p_low.sum():.1f}")
print(f"Weighted entries with 0.5 < -t < 1.0: {p_high.sum():.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
xlabel = r"M($\eta\pi^+$) (GeV/$c^2$)"

# pcolormesh wants [y, x], so transpose with .T; vmin=0 hides negative bins
mesh = ax1.pcolormesh(x_edges, y_edges, h2.T, cmap="viridis", vmin=0)
fig.colorbar(mesh, ax=ax1, label="Weighted entries")
ax1.set_title(r"$\eta\pi^+$ mass vs $-t$")
ax1.set_xlabel(xlabel)
ax1.set_ylabel(r"$-t$ (GeV$^2$)")

ax2.stairs(p_low, x_edges, color="tab:blue", lw=1.5, label=r"0 < $-t$ < 0.5 GeV$^2$")
ax2.stairs(p_high, x_edges, color="tab:red", lw=1.5, label=r"0.5 < $-t$ < 1.0 GeV$^2$")
ax2.set_title("Projections onto the mass axis")
ax2.set_xlabel(xlabel)
ax2.set_ylabel("Weighted entries / 20 MeV")
ax2.set_ylim(0, 1.3 * max(p_low.max(), p_high.max()))
ax2.legend(loc="upper right")

fig.tight_layout()
fig.savefig("plots/py_ex5_2d.png", dpi=100)
print("Saved plots/py_ex5_2d.png")
plt.show()

## Exercise 6: Fit the a$_0$(980) peak

Gaussian + straight line between 0.88 and 1.10 GeV. Change `fit_lo`, `fit_hi` and rerun.

In [ ]:
fit_lo = 0.88    # fit range in GeV: try 0.85 and 1.15
fit_hi = 1.10

tree = uproot.open(filename)["pipneta__B4"]
d = tree.arrays(["PiPlus_Px", "PiPlus_Py", "PiPlus_Pz", "PiPlus_E",
                 "Eta_Px", "Eta_Py", "Eta_Pz", "Eta_E",
                 "Chi2_NDF", "MissingMassSquared", "Mandelstam_t", "accidweight"],
                library="np")

E  = d["PiPlus_E"]  + d["Eta_E"]
px = d["PiPlus_Px"] + d["Eta_Px"]
py = d["PiPlus_Py"] + d["Eta_Py"]
pz = d["PiPlus_Pz"] + d["Eta_Pz"]
mass = np.sqrt(E**2 - px**2 - py**2 - pz**2)
w = d["accidweight"]
keep = ((d["Chi2_NDF"] < 10) & (-d["Mandelstam_t"] < 1.0)
        & (np.abs(d["MissingMassSquared"]) < 0.05))

edges = np.linspace(0.7, 2.5, 181)                    # 180 bins of 10 MeV
counts, _ = np.histogram(mass[keep], bins=edges, weights=w[keep])
sumw2, _ = np.histogram(mass[keep], bins=edges, weights=w[keep] ** 2)
errors = np.sqrt(sumw2)                               # error on a weighted bin
centres = 0.5 * (edges[1:] + edges[:-1])


def model(x, amp, mean, sigma, b0, b1):
    """Gaussian peak plus a straight-line background."""
    return amp * np.exp(-0.5 * ((x - mean) / sigma) ** 2) + b0 + b1 * x


sel = (centres > fit_lo) & (centres < fit_hi)         # bins inside the fit range

# Starting values: height ~500 above background at ~0.99 GeV, ~30 MeV wide.
start = [500, 0.99, 0.03, 450, 0]
params, cov = curve_fit(model, centres[sel], counts[sel], p0=start,
                        sigma=errors[sel], absolute_sigma=True)
uncert = np.sqrt(np.diag(cov))
amp, mean, sigma, b0, b1 = params

chi2 = np.sum(((counts[sel] - model(centres[sel], *params)) / errors[sel]) ** 2)
ndf = sel.sum() - len(params)

print("===== Summary =====")
print(f"Mean     = {1000 * mean:.2f} +/- {1000 * uncert[1]:.2f} MeV")
print(f"Sigma    = {1000 * abs(sigma):.2f} +/- {1000 * uncert[2]:.2f} MeV")
print(f"chi2/ndf = {chi2:.2f} / {ndf}  (p-value {chi2_dist.sf(chi2, ndf):.4f})")
# Area of a Gaussian = amp * sigma * sqrt(2 pi); divide by bin width for a count
yield_ = amp * abs(sigma) * np.sqrt(2 * np.pi) / (edges[1] - edges[0])
print(f"Signal yield ~ {yield_:.0f} (weighted entries)")

fig, ax = plt.subplots(figsize=(8, 6))
show = (centres > 0.7) & (centres < 1.6)
ax.errorbar(centres[show], counts[show], yerr=errors[show], fmt="o", ms=3,
            color="black", label="Data")
x = np.linspace(fit_lo, fit_hi, 400)
ax.plot(x, model(x, *params), color="tab:red", lw=2, label="Gaussian + line")
ax.plot(x, b0 + b1 * x, color="gray", ls="--", label="Background only")
ax.set_title(r"a$_0$(980) $\rightarrow\eta\pi^+$ fit")
ax.set_xlabel(r"M($\eta\pi^+$) (GeV/$c^2$)")
ax.set_ylabel("Weighted entries / 10 MeV")
ax.set_ylim(0, 1.6 * counts[show].max())
ax.legend(loc="upper left")
ax.text(0.97, 0.97,
        f"mean = {1000 * mean:.1f} ± {1000 * uncert[1]:.1f} MeV\n"
        f"σ = {1000 * abs(sigma):.1f} ± {1000 * uncert[2]:.1f} MeV\n"
        f"χ²/ndf = {chi2:.1f} / {ndf}",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(facecolor="white", edgecolor="gray"))
fig.tight_layout()
fig.savefig("plots/py_ex6_fit.png", dpi=100)
print("Saved plots/py_ex6_fit.png")
plt.show()

## Exercise 7: Two-dimensional mass plots

The three two-body masses against each other, a Dalitz plot, and the effect of a baryon veto on M($\eta\pi^+$). Mesons make horizontal bands, baryons vertical ones.

In [ ]:
tree = uproot.open(filename)["pipneta__B4"]
d = tree.arrays(["PiPlus_Px", "PiPlus_Py", "PiPlus_Pz", "PiPlus_E",
                 "Eta_Px", "Eta_Py", "Eta_Pz", "Eta_E",
                 "Neutron_Px", "Neutron_Py", "Neutron_Pz", "Neutron_E",
                 "Chi2_NDF", "Mandelstam_t", "MissingMassSquared", "accidweight"],
                library="np")

# Standard cuts, then keep only the entries that pass
keep = ((d["Chi2_NDF"] < 10) & (-d["Mandelstam_t"] < 1.0)
        & (np.abs(d["MissingMassSquared"]) < 0.05))
d = {k: v[keep] for k, v in d.items()}
w = d["accidweight"]


def p4(name):
    """(E, px, py, pz) of one particle as four numpy arrays."""
    return np.array([d[name + "_E"], d[name + "_Px"], d[name + "_Py"], d[name + "_Pz"]])


def mass2(a, b):
    """Invariant mass squared of two four-vectors: (E1+E2)^2 - |p1+p2|^2."""
    s = a + b
    return s[0] ** 2 - s[1] ** 2 - s[2] ** 2 - s[3] ** 2


pip, eta, neu = p4("PiPlus"), p4("Eta"), p4("Neutron")
m2_etapi, m2_etan, m2_pin = mass2(eta, pip), mass2(eta, neu), mass2(pip, neu)
m_etapi, m_etan, m_pin = np.sqrt(m2_etapi), np.sqrt(m2_etan), np.sqrt(m2_pin)

panels = [  # (x, y, x bins, y bins, x label, y label, title)
    (m_etan, m_etapi, np.linspace(1.4, 4.4, 101), np.linspace(0.7, 2.5, 91),
     r"M($\eta n$) (GeV/$c^2$)", r"M($\eta\pi^+$) (GeV/$c^2$)", r"M($\eta\pi^+$) vs M($\eta n$)"),
    (m_pin, m_etapi, np.linspace(1.0, 4.0, 101), np.linspace(0.7, 2.5, 91),
     r"M($\pi^+n$) (GeV/$c^2$)", r"M($\eta\pi^+$) (GeV/$c^2$)", r"M($\eta\pi^+$) vs M($\pi^+n$)"),
    (m_pin, m_etan, np.linspace(1.0, 4.0, 101), np.linspace(1.4, 4.4, 101),
     r"M($\pi^+n$) (GeV/$c^2$)", r"M($\eta n$) (GeV/$c^2$)", r"M($\eta n$) vs M($\pi^+n$)"),
    (m2_pin, m2_etapi, np.linspace(1.0, 16.0, 101), np.linspace(0.0, 9.0, 101),
     r"M$^2$($\pi^+n$) (GeV$^2$/$c^4$)", r"M$^2$($\eta\pi^+$) (GeV$^2$/$c^4$)", "Dalitz plot"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (x, y, xe, ye, xl, yl, title) in zip(axes.flat, panels):
    h, _, _ = np.histogram2d(x, y, bins=[xe, ye], weights=w)
    h = np.where(h < 0.1, np.nan, h)    # leave empty / negative bins white, like SetMinimum(0.1)
    mesh = ax.pcolormesh(xe, ye, h.T, cmap="viridis")
    fig.colorbar(mesh, ax=ax)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(title)
    if title == "Dalitz plot":
        print(f"Weighted entries in the Dalitz plot:          {np.nansum(np.histogram2d(x, y, bins=[xe, ye], weights=w)[0]):.1f}")
fig.tight_layout()
fig.savefig("plots/py_ex7_masses2d.png", dpi=100)

# Baryon veto
edges = np.linspace(0.7, 2.5, 91)
veto = (m_pin > 2.0) & (m_etan > 2.0)
h_all, _ = np.histogram(m_etapi, bins=edges, weights=w)
h_veto, _ = np.histogram(m_etapi[veto], bins=edges, weights=w[veto])
print(f"M(eta pi+) 0.7-2.5 GeV, standard cuts:        {h_all.sum():.1f}")
print(f"M(eta pi+) 0.7-2.5 GeV, plus baryon veto:     {h_veto.sum():.1f}")
print(f"Fraction kept by the veto:                    {h_veto.sum() / h_all.sum():.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.stairs(h_all, edges, color="black", lw=1.5, label="Standard cuts")
ax.stairs(h_veto, edges, color="tab:red", lw=1.5, label=r"+ M($\pi^+n$) > 2 and M($\eta n$) > 2 GeV")
ax.set_title("Baryon veto")
ax.set_xlabel(r"M($\eta\pi^+$) (GeV/$c^2$)")
ax.set_ylabel("Weighted entries / 20 MeV")
ax.set_ylim(0, 1.3 * h_all.max())
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig("plots/py_ex7_veto.png", dpi=100)
print("Saved plots/py_ex7_masses2d.png and plots/py_ex7_veto.png")
plt.show()

## Exercise 8: Gottfried-Jackson angles

Boost lab $	o$ CM $	o$ $\eta\pi^+$ rest frame, build the GJ axes, and plot $\cos	heta_{GJ}$ and $\phi_{GJ}$ of the $\eta$. A cross-check computes $\cos	heta_{GJ}$ a second way.

In [ ]:
tree = uproot.open(filename)["pipneta__B4"]
d = tree.arrays(["PiPlus_Px", "PiPlus_Py", "PiPlus_Pz", "PiPlus_E",
                 "Eta_Px", "Eta_Py", "Eta_Pz", "Eta_E",
                 "Beam_Px", "Beam_Py", "Beam_Pz", "Beam_E",
                 "Chi2_NDF", "Mandelstam_t", "MissingMassSquared", "accidweight"],
                library="np")
keep = ((d["Chi2_NDF"] < 10) & (-d["Mandelstam_t"] < 1.0)
        & (np.abs(d["MissingMassSquared"]) < 0.05))
d = {k: v[keep] for k, v in d.items()}
w = d["accidweight"]


def p4(name):
    """(E, px, py, pz) of one particle as a 4 x N array."""
    return np.array([d[name + "_E"], d[name + "_Px"], d[name + "_Py"], d[name + "_Pz"]])


def boost(v, b):
    """Boost four-vectors v (4 x N) by velocity b (3 x N), same formula as TLorentzVector::Boost."""
    E, p = v[0], v[1:]
    b2 = np.sum(b * b, axis=0)
    gamma = 1.0 / np.sqrt(1.0 - b2)
    bp = np.sum(b * p, axis=0)
    gamma2 = np.where(b2 > 0, (gamma - 1.0) / np.where(b2 > 0, b2, 1.0), 0.0)
    p_new = p + gamma2 * bp * b + gamma * b * E
    E_new = gamma * (E + bp)
    return np.vstack([E_new, p_new])


def to_rest_frame_of(obj):
    """Velocity that takes you to the rest frame of obj: -p/E."""
    return -obj[1:] / obj[0]


def unit(v):
    return v / np.linalg.norm(v, axis=0)


beam, pip, eta = p4("Beam"), p4("PiPlus"), p4("Eta")
etapi = eta + pip
m_etapi = np.sqrt(etapi[0] ** 2 - np.sum(etapi[1:] ** 2, axis=0))
target = np.zeros_like(beam)
target[0] = 0.938272                      # proton at rest

# Step 1: lab -> CM
b_cm = to_rest_frame_of(beam + target)
beam_cm, eta_cm, etapi_cm = boost(beam, b_cm), boost(eta, b_cm), boost(etapi, b_cm)

# Step 2: CM -> eta pi+ rest frame
b_gj = to_rest_frame_of(etapi_cm)
beam_gj, eta_gj = boost(beam_cm, b_gj), boost(eta_cm, b_gj)

# Step 3: axes
z = unit(beam_gj[1:])
y = unit(np.cross(beam_cm[1:], etapi_cm[1:], axis=0))
x = np.cross(y, z, axis=0)

# Step 4: eta direction in (x, y, z)
vx, vy, vz = (np.sum(eta_gj[1:] * a, axis=0) for a in (x, y, z))
cos_theta = vz / np.sqrt(vx ** 2 + vy ** 2 + vz ** 2)
phi = np.degrees(np.arctan2(vy, vx))

# Cross-check: straight from the lab to the eta pi+ rest frame
b_direct = to_rest_frame_of(etapi)
cos_check = np.sum(unit(boost(eta, b_direct)[1:]) * unit(boost(beam, b_direct)[1:]), axis=0)

a0 = (m_etapi > 0.92) & (m_etapi < 1.04)
a2 = (m_etapi > 1.24) & (m_etapi < 1.40)
print(f"Entries used:                    {len(w)}")
print(f"cos(theta_GJ) range:             {cos_theta.min():.6f} to {cos_theta.max():.6f}")
print(f"phi_GJ range (deg):              {phi.min():.3f} to {phi.max():.3f}")
print(f"Max |cos - cross-check|:         {np.max(np.abs(cos_theta - cos_check)):.3g}")
in_range = (cos_theta >= -1) & (cos_theta < 1)
print(f"Weighted entries, all masses:    {w[in_range].sum():.1f}")
print(f"Weighted entries, a0 window:     {w[a0].sum():.2f}  (mean cos = {np.average(cos_theta[a0], weights=w[a0]):.4f})")
print(f"Weighted entries, a2 window:     {w[a2].sum():.2f}  (mean cos = {np.average(cos_theta[a2], weights=w[a2]):.4f})")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ce = np.linspace(-1, 1, 51)
pe = np.linspace(-180, 180, 61)

ax = axes[0, 0]
ax.stairs(np.histogram(cos_theta, ce, weights=w)[0], ce, lw=1.5)
ax.set_title(r"cos$\theta_{GJ}$ (all masses)")
ax.set_xlabel(r"cos$\theta_{GJ}$")
ax.set_ylabel("Weighted entries")
ax.set_ylim(0, None)

ax = axes[0, 1]
ax.stairs(np.histogram(phi, pe, weights=w)[0], pe, lw=1.5)
ax.set_title(r"$\phi_{GJ}$ (all masses)")
ax.set_xlabel(r"$\phi_{GJ}$ (deg)")
ax.set_ylabel("Weighted entries")
ax.set_ylim(0, None)

ax = axes[1, 0]
me = np.linspace(0.7, 2.5, 46)
ce2 = np.linspace(-1, 1, 41)
h, _, _ = np.histogram2d(m_etapi, cos_theta, bins=[me, ce2], weights=w)
mesh = ax.pcolormesh(me, ce2, np.where(h < 0.1, np.nan, h).T, cmap="viridis")
fig.colorbar(mesh, ax=ax)
ax.set_title(r"cos$\theta_{GJ}$ vs M($\eta\pi^+$)")
ax.set_xlabel(r"M($\eta\pi^+$) (GeV/$c^2$)")
ax.set_ylabel(r"cos$\theta_{GJ}$")

ax = axes[1, 1]
cw = np.linspace(-1, 1, 26)
h0 = np.histogram(cos_theta[a0], cw, weights=w[a0])[0]
h2 = np.histogram(cos_theta[a2], cw, weights=w[a2])[0]
ax.stairs(h0 / h0.sum(), cw, color="tab:blue", lw=1.5, label="a$_0$(980): 0.92-1.04 GeV")
ax.stairs(h2 / h2.sum(), cw, color="tab:red", lw=1.5, label="a$_2$(1320): 1.24-1.40 GeV")
ax.set_title(r"cos$\theta_{GJ}$ in mass windows")
ax.set_xlabel(r"cos$\theta_{GJ}$")
ax.set_ylabel("Fraction / 0.08")
ax.set_ylim(0, 1.5 * max(h0.max() / h0.sum(), h2.max() / h2.sum()))
ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig("plots/py_ex8_gj_angles.png", dpi=100)
print("Saved plots/py_ex8_gj_angles.png")
plt.show()

## Next

Compare your numbers with `reference/expected_output.txt` and your plots with `reference/py_*.png`, then try the challenges in `README.md`.